# 📊 **Análisis Exploratorio de Datos y Fundamentos de NumPy**

En este notebook, en una primera parte, trabajaremos el análisis exploratorio de datos con `Pandas` usando el dataset *California Housing*. En la segunda parte nos concentraremos en `NumPy` desde la perspectiva del manejo de arreglos, matrices y operaciones vectorizadas, como preparación para temas posteriores de Machine Learning.

### 🔜 **Introducción**

En esta sesión se conectan dos ideas fundamentales para el trabajo en Ciencia de Datos: la exploración estructurada de datos tabulares con Pandas y el dominio de arreglos numéricos con NumPy. Esta combinación será clave para comprender más adelante transformaciones, preprocesamiento y algoritmos de Machine Learning.

##### 🎯 **Objetivos de la sesión**

- Aplicar técnicas de análisis exploratorio de datos con `Pandas` sobre un dataset real.
- Interpretar agrupaciones, tablas cruzadas, distribuciones y relaciones entre variables.
- Comprender la estructura de los `ndarray` de NumPy y su diferencia frente a listas de Python.
- Practicar operaciones vectorizadas, indexación, *broadcasting* y funciones matriciales básicas.
- Preparar las bases conceptuales necesarias para módulos posteriores de Machine Learning.

##### 🧭 **¿Qué veremos?**

Primero exploraremos el dataset `california_housing_test.csv` con herramientas de `Pandas`: 
* inspección de estructura
* estadísticos descriptivos
* creación de categorías
* agrupaciones 
* tablas cruzadas, correlaciones 
* construcción de variables derivadas (feeature engineering).  

Luego cambiaremos de enfoque para estudiar `NumPy` como biblioteca central para computación numérica eficiente, trabajando con arreglos, matrices, transformaciones, agregaciones y operaciones vectorizadas.

In [ ]:
import pandas as pd

df = pd.read_csv('sample_data/california_housing_test.csv')
df.head()

### 📊 **EDA con Pandas**

Realizaremos un análisis exploratorio, no sólo para describir el dataset, sino también para empezar a formular preguntas analíticas sobre su estructura y sobre posibles patrones entre las variables.

##### 🧱 **Estructura general del dataset**

Antes de analizar relaciones entre variables, conviene revisar el tipo de datos, la presencia de valores faltantes y la forma general de la tabla.

In [ ]:
df.info()

##### 📌 **Estadísticos descriptivos**

Esta exploración resume tendencia central, dispersión y rangos. Es útil para detectar escalas muy distintas entre variables y posibles valores extremos.

In [ ]:
df.describe().T

##### 🧮 **Cuantiles y dispersión**

Los cuantiles permiten entender mejor la distribución de cada variable más allá del promedio.

In [ ]:
df.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T

##### 🏷️ **Creación de categorías por ingreso**

La discretización de variables continuas es útil para construir grupos comparables y facilitar análisis agregados.

In [ ]:
df['income_cat'] = pd.cut(
    df['median_income'],
    bins=[0, 2, 4, 6, 8, df['median_income'].max()],
    labels=['Muy bajo', 'Bajo', 'Medio', 'Alto', 'Muy alto'],
    include_lowest=True
)
df[['median_income', 'income_cat']].head(10)

##### 🔁 **GroupBy avanzado**

Agrupamos por categoría de ingreso para comparar el comportamiento del valor mediano de la vivienda, el tamaño de la población y la cantidad promedio de habitaciones.

In [ ]:
df.groupby('income_cat').agg({
    'median_house_value': ['mean', 'median', 'min', 'max'],
    'population': ['mean', 'sum'],
    'total_rooms': ['mean'],
    'households': ['mean']
}).round(2)

##### 🔀 **Tablas cruzadas con crosstab**

Las tablas cruzadas ayudan a explorar la relación entre variables categorizadas. Aquí cruzamos categorías de ingreso con rangos de edad de las viviendas.

In [ ]:
df['age_cat'] = pd.cut(
    df['housing_median_age'],
    bins=[0, 10, 20, 30, 40, 60],
    labels=['0-10', '11-20', '21-30', '31-40', '41-60'],
    include_lowest=True
)

pd.crosstab(df['age_cat'], df['income_cat'], margins=True)

##### 📍 **Tablas cruzadas normalizadas**

Normalizar una tabla cruzada permite interpretar proporciones y no sólo conteos absolutos.

In [ ]:
pd.crosstab(df['age_cat'], df['income_cat'], normalize='index').round(3)

##### 🧠 **Feature Engineering**

Construimos variables derivadas frecuentemente usadas en análisis habitacional y modelado.

In [ ]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head()

##### 📈 **Correlación entre variables numéricas**

La matriz de correlación ofrece una primera aproximación a relaciones lineales entre variables.

In [ ]:
corr_matrix = df.corr(numeric_only=True)
corr_matrix['median_house_value'].sort_values(ascending=False)

##### 🥇 **Top registros por valor de vivienda**

Ordenar registros permite inspeccionar ejemplos extremos y contrastarlos con otras características.

In [ ]:
df.nlargest(10, 'median_house_value')[['longitude', 'latitude', 'median_income', 'median_house_value', 'rooms_per_household']]

##### 🪞 **Consulta analítica con query**

El método `query` permite escribir filtros complejos de forma legible.

In [ ]:
df.query('median_income > 6 and housing_median_age >= 25')[['median_income', 'housing_median_age', 'median_house_value']].head(10)

##### 🧰 **Tabla resumen con pivot_table**

Una tabla dinámica permite resumir el valor mediano de la vivienda según combinaciones de categorías de ingreso y edad.

In [ ]:
pd.pivot_table(
    df,
    values='median_house_value',
    index='income_cat',
    columns='age_cat',
    aggfunc='mean'
).round(2)

### 🔢 **Introducción a NumPy**

En esta segunda parte el objetivo es concentrarnos en el manejo de arreglos de `NumPy`, su lógica de indexación, las operaciones vectorizadas y las funciones matriciales básicas. 

Esto es esencial porque gran parte del ecosistema de Machine Learning trabaja internamente sobre estructuras tipo `ndarray`.

##### 🚀 **¿Por qué NumPy es importante?**

`NumPy` permite trabajar con datos homogéneos en memoria de forma mucho más eficiente que con listas de Python, especialmente cuando se realizan operaciones matemáticas sobre grandes volúmenes de datos. 

Su modelo de cálculo vectorizado evita muchos ciclos explícitos y sirve de base para bibliotecas como pandas, scikit-learn y muchas herramientas de álgebra lineal y optimización.

In [ ]:
import numpy as np

##### 🧱 **Creación de arrays**

Comenzamos creando arreglos de una y dos dimensiones para observar su estructura interna.

In [ ]:
a = np.array([2, 4, 6, 8, 10])
B = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

a, B

##### 🔍 **Dimensiones, forma y tipo de dato**

Estas propiedades son fundamentales para entender cómo NumPy almacena y procesa la información.

In [ ]:
print('a.ndim =', a.ndim)
print('a.shape =', a.shape)
print('a.dtype =', a.dtype)
print('B.ndim =', B.ndim)
print('B.shape =', B.shape)
print('B.dtype =', B.dtype)

##### 🛠️ **Funciones de inicialización útiles**

Con frecuencia necesitaremos arreglos de ceros, unos, secuencias o matrices identidad.

In [ ]:
ceros = np.zeros((2, 4))
unos = np.ones((3, 2))
secuencia = np.arange(0, 12, 2)
espaciado = np.linspace(0, 1, 5)
identidad = np.eye(4)

ceros, unos, secuencia, espaciado, identidad

##### ✂️ **Indexación y slicing**

El acceso a subconjuntos de un arreglo es una habilidad básica y muy usada en análisis numérico.

In [ ]:
M = np.array([[10, 20, 30, 40],
              [50, 60, 70, 80],
              [90, 100, 110, 120]])

print(M)
print('Elemento [1, 2]:', M[1, 2])
print('Primera fila:', M[0, :])
print('Segunda columna:', M[:, 1])
print('Submatriz:\n', M[:2, 1:3])

##### 🔄 **Reshape y flatten**

Modificar la forma de un arreglo sin cambiar sus datos es una operación muy frecuente en Machine Learning.

In [ ]:
x = np.arange(12)
X = x.reshape(3, 4)
x_plano = X.flatten()

print('x =', x)
print('X =\n', X)
print('x_plano =', x_plano)

##### ⚡ **Operaciones vectorizadas**

NumPy permite aplicar operaciones a todos los elementos del arreglo al mismo tiempo, sin usar bucles explícitos.

In [ ]:
v = np.array([1, 2, 3, 4])

print('v + 10 =', v + 10)
print('v * 3 =', v * 3)
print('v ** 2 =', v ** 2)
print('np.sqrt(v) =', np.sqrt(v))

##### 📏 **Agregaciones básicas**

Las funciones de agregación sintetizan información global o por ejes.

In [ ]:
N = np.array([[3, 5, 7],
              [2, 4, 6],
              [1, 8, 9]])

print('Suma total:', np.sum(N))
print('Promedio total:', np.mean(N))
print('Máximo por columna:', np.max(N, axis=0))
print('Suma por fila:', np.sum(N, axis=1))

##### 🎯 **Broadcasting**

El *broadcasting* permite combinar arreglos de formas compatibles sin replicar datos manualmente.

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
b = np.array([100, 200, 300])

A + b

##### 🧪 **Comparaciones y máscaras booleanas**

Las máscaras booleanas son clave para filtrar datos de forma eficiente.

In [ ]:
z = np.array([4, 7, 1, 9, 2, 8, 3])
mascara = z > 5

print('Máscara:', mascara)
print('Elementos mayores que 5:', z[mascara])

##### 🧮 **Operaciones matriciales básicas**

NumPy distingue entre multiplicación elemento a elemento y multiplicación matricial.

In [ ]:
P = np.array([[1, 2],
              [3, 4]])
Q = np.array([[5, 6],
              [7, 8]])

print('Producto elemento a elemento:\n', P * Q)
print('Producto matricial con @:\n', P @ Q)

##### 📐 **Transpuesta, determinante e inversa**

Estas operaciones aparecen frecuentemente en álgebra lineal aplicada a modelos de Machine Learning.

In [ ]:
R = np.array([[2, 1],
              [5, 3]])

print('Transpuesta:\n', R.T)
print('Determinante:', np.linalg.det(R))
print('Inversa:\n', np.linalg.inv(R))

##### 🧭 **Generación pseudoaleatoria y reproducibilidad**

En experimentos numéricos y entrenamiento de modelos es importante poder reproducir resultados.

In [ ]:
np.random.seed(42)
aleatorio = np.random.randint(0, 10, size=(3, 4))
aleatorio